# Experiment 2: Data Cleaning of a Clinical Trial Dataset (Python)
**Course:** 26AI2003 – Statistical Programming for Clinical Data Analytics

## Aim
To detect and correct data-quality problems in a raw clinical trial dataset (missing values, duplicates, inconsistent text, invalid values and mixed date formats) and produce a clean, analysis-ready dataset with a record of every change made.

## Description
The raw file `clinical_trial_data_raw.csv` has 193 rows. It is the 180-patient dataset with deliberately added problems: 8 exact duplicate rows, 5 repeated patient IDs with a different HEARTRATE, messy `SEX` and `ARM` text, impossible `AGE` values, four different `ENROLL_DATE` formats, and blank cells. The notebook checks each problem first (**before**), fixes it step by step, and finally confirms that the result has 180 rows and no missing values (**after**).

**Needs:** `pandas` only. Keep the CSV in the same folder as this notebook.

## Step 1: Import and look at the raw data

In [ ]:
import pandas as pd

df = pd.read_csv("clinical_trial_data_raw.csv")

print("Rows, columns:", df.shape)
df.head()

## Step 2: Problems BEFORE cleaning

In [ ]:
print("Exact duplicate rows      :", df.duplicated().sum())
print("Duplicate USUBJID values  :", df.duplicated(subset="USUBJID").sum())
print()
print("SEX values:", sorted(df["SEX"].dropna().unique().tolist()))
print("ARM values:", sorted(df["ARM"].dropna().unique().tolist()))
print()
print("Invalid AGE (<18 or >100) :", ((df["AGE"] < 18) | (df["AGE"] > 100)).sum())
print()
print("Missing values per column:")
print(df.isna().sum())

## Step 3: Remove exact duplicate rows
Every column matches another row, so this is a double data entry. Keep one copy.

In [ ]:
before = len(df)
df = df.drop_duplicates()
print("Rows removed:", before - len(df))

## Step 4: Remove duplicate patient IDs (conflicting data)
The same `USUBJID` appears twice with a different `HEARTRATE`. Rule for this lab: **keep the first occurrence**. In a real study the discrepancy would be traced back to the source record.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="USUBJID", keep="first")
print("Rows removed:", before - len(df))
print("Rows now    :", len(df))

## Step 5: Standardize SEX and ARM
Trim spaces, ignore case, then map every variant to one standard value.

In [ ]:
# SEX: "Male", "m", " M " -> "M" ; "Female", "f" -> "F"
sex = df["SEX"].str.strip().str.upper()
df["SEX"] = sex.map({"M": "M", "MALE": "M", "F": "F", "FEMALE": "F"})

# ARM: remove all spaces and ignore case, then look up the standard name
arm = df["ARM"].str.replace(" ", "", regex=False).str.upper()
df["ARM"] = arm.map({"PLACEBO": "Placebo", "DRUG10MG": "Drug 10mg", "DRUG20MG": "Drug 20mg"})

print("SEX values:", df["SEX"].unique().tolist())
print("ARM values:", df["ARM"].unique().tolist())

## Step 6: Standardize ENROLL_DATE
Four formats are present. Each date is tried against every format and rewritten as `YYYY-MM-DD`.

*Assumption:* a slash date such as `04/25/2025` is month/day/year and `25/04/2025` is day/month/year. This works here because the day is above 12 whenever a slash date is used.

In [ ]:
formats = ["%d-%b-%Y", "%Y-%m-%d", "%m/%d/%Y", "%d/%m/%Y"]

clean_dates = pd.Series(pd.NaT, index=df.index)
for f in formats:
    parsed = pd.to_datetime(df["ENROLL_DATE"], format=f, errors="coerce")
    clean_dates = clean_dates.fillna(parsed)

print("Dates that could not be read:", clean_dates.isna().sum())
df["ENROLL_DATE"] = clean_dates.dt.strftime("%Y-%m-%d")
df["ENROLL_DATE"].head()

## Step 7: Fix invalid AGE values
An adult patient cannot be aged 999, -5, 0 or 150. These are data-entry errors, so they are set to missing and filled in Step 8.

In [ ]:
bad_age = (df["AGE"] < 18) | (df["AGE"] > 100)
print("Invalid ages set to missing:", bad_age.sum())
df.loc[bad_age, "AGE"] = None

## Step 8: Handle missing values (imputation)
- **Numeric columns** → filled with the **median** (not affected by outliers).
- **SITEID** (categorical) → filled with the label **"Unknown"**, so an imputed value is never mistaken for a real one.

The printout below is the audit trail.

In [ ]:
numeric_cols = ["AGE", "WEIGHT_KG", "BMI", "GLUCOSE_MGDL", "CHOLESTEROL_MGDL"]

for col in numeric_cols:
    n_missing = df[col].isna().sum()
    if n_missing > 0:
        median = df[col].median()
        df[col] = df[col].fillna(median)
        print(f"{col}: filled {n_missing} missing values with median {median:.1f}")

n_site = df["SITEID"].isna().sum()
df["SITEID"] = df["SITEID"].fillna("Unknown")
print(f"SITEID: filled {n_site} missing values with 'Unknown'")

## Step 9: Check AFTER cleaning

In [ ]:
print("Rows, columns            :", df.shape)
print("Exact duplicate rows     :", df.duplicated().sum())
print("Duplicate USUBJID values :", df.duplicated(subset="USUBJID").sum())
print("Total missing values     :", df.isna().sum().sum())
print("SEX values               :", sorted(df["SEX"].unique().tolist()))
print("ARM values               :", sorted(df["ARM"].unique().tolist()))
print("AGE range                :", df["AGE"].min(), "to", df["AGE"].max())

assert len(df) == 180, "Expected 180 rows"
assert df.isna().sum().sum() == 0, "Missing values remain"
print("\nAll checks passed.")

## Step 10: Save the clean dataset

In [ ]:
df.to_csv("clinical_trial_data_clean.csv", index=False)
print("Saved: clinical_trial_data_clean.csv")
df.head()